<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/XML%20for%20Contracts/07-lab-xml-contracts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 7 Lab — XML for Contracts: Schema Validation (XSD) and XPath Extraction

This lab applies the **parse → validate → extract** chain to a simplified e-invoicing scenario. You read an XSD contract by filling in element contract cards, predict validation outcomes before running the validator, compare XSD and DTD validation, extract fields with XPath, validate JSON with JSON Schema, and complete a cross-model comparison.

Use the formal vocabulary (well-formed, valid, element, attribute, compositor, cardinality, contract card, XPath, axis, node test, predicate, namespace, validator rejects) throughout your work.

**Time budget:** Part 1 ~20 min | Part 2 ~25 min | Part 3 ~10 min | Part 4 ~25 min | Part 5 ~15 min | Part 6 ~15 min | **Total ~110 min**

**Tools required:** Google Colab with `cellspell` (`%%xpath` magic for xmllint, Python for JSON Schema)

---

## Setup

Run the following cells in a new Google Colab notebook. All XML, XSD, and DTD files are created via `%%writefile` cells — you do not need to upload anything.

**Cell 1 — Install cellspell:**

In [ ]:
# Topic 7 — XML for Contracts: Schema Validation and XPath Extraction
!pip install "cellspell[xpath] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
%load_ext cellspell.xpath


**Cell 2 — Verify xmllint:**

In [ ]:
!xmllint --version 2>&1 | head -1
# Expected: xmllint using libxml version ...


**Cell 3 — Download XML contract files from course repository:**


In [ ]:
# ── Download XML/XSD/DTD files from course repository ──────
import urllib.request, os

BASE_URL = "https://raw.githubusercontent.com/sreent/data-management-intro/main/resources/e-invoicing/"
FILES = ["invoice_schema.xsd", "invoice_valid.xml", "invoice_invalid.xml",
         "invoice.dtd", "invoice_namespaced.xml"]

for fname in FILES:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(BASE_URL + fname, fname)
        print(f"Downloaded {fname} ({os.path.getsize(fname):,} bytes)")

print("All contract files ready.")


---

## Part 1 — Read the Contract (XSD Comprehension)

Before validating any XML, read the XSD. This part builds the skill from Requirement 2: systematic contract reading via element contract cards.

### Q1. Contract card — `Invoice` (root element)

Read `invoice_schema.xsd` and fill in the contract card for the root `Invoice` element:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children (in order) | |
| Restrictions | |
| Attributes | |

**Hint:** The root element uses an anonymous complex type (defined inline). List all children declared in the `xs:sequence`, including their cardinalities.

### Q2. Contract card — `InvoiceLine`

Fill in the contract card for `InvoiceLine`:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children (in order) | |
| Restrictions | |
| Attributes | |

### Q3. Contract card — `Quantity`

Fill in the contract card for `Quantity`:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children | |
| Restrictions | |
| Attributes | |

**Key question:** What values does `xs:positiveInteger` accept? Would `0` pass? Would `-3` pass? Would `5.5` pass?

### Q4. Contract card — `PaymentMethodCode`

Fill in the contract card for `PaymentMethodCode`:

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children | |
| Restrictions | |
| Attributes | |

**Key question:** List the four allowed values. Would `"bank_transfer"` pass? Would `"30 "` (with trailing space) pass?

### Q5. Contract card — `LineAmount`

Fill in the contract card for `LineAmount`. This element has both text content (a decimal) and an attribute (`currencyID`).

| Field | Your answer |
|---|---|
| Name | |
| Parent | |
| Compositor | |
| Cardinality | |
| Type | |
| Children | |
| Restrictions | |
| Attributes (name, type, required?) | |

**Key question:** `AmountType` uses `xs:simpleContent` with `xs:extension`. What does this mean structurally? How does it differ from a plain `xs:string` element?

---

## Part 2 — Validate with XSD (Predict, Then Verify)

### Q6. Well-formedness check — both documents

Run the well-formedness check on both documents. **Predict** the result before running:

In [ ]:
%xpath invoice_valid.xml

In [ ]:
%xpath invoice_invalid.xml


**Record:** Are both well-formed? Why would an invalid document still be well-formed?

### Q7. Predict validation failures (the four-check model)

**Before running the validator**, read `invoice_invalid.xml` alongside the XSD. Using the four-check model from the narrative (name, compositor, cardinality, content), identify each error.

Fill in the table:

| Error # | What's wrong | XSD rule violated | Which check? (name / compositor / cardinality / content) |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

**Important:** Complete this table entirely from reading the XSD and XML — do not run the validator yet.

### Q8. Validate and compare

Now validate the invalid document:

In [ ]:
%xpath --xsd invoice_schema.xsd invoice_invalid.xml


**Record the actual errors.** Compare to your predictions in Q7:

- Which errors did you correctly predict?
- Did the validator report errors in the same order as your predictions?
- Did any error cascade (one error causing the validator to report additional errors)?

### Q9. Validate the valid document

Confirm that the valid invoice passes:

In [ ]:
%xpath --xsd invoice_schema.xsd invoice_valid.xml


**Expected:** `✓ Valid`. If this fails, check the XSD and XML files from the Setup cells.

---

## Part 3 — DTD Comparison (What DTD Cannot Express)

### Q10. Validate with DTD — invalid document

Validate the invalid invoice against the DTD:

In [ ]:
%xpath --dtd invoice.dtd invoice_invalid.xml


**Before running, predict:** Which of the four errors from Q7 will the DTD catch? Which will it miss? Why?

**Record the actual DTD output.** Fill in:

| Error # | XSD catches? | DTD catches? | Why DTD misses it (if applicable) |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

### Q11. DTD limitation analysis

In 2–3 sentences, explain: Why do standards bodies (UBL, XBRL, HL7) use XSD instead of DTD? Reference the specific errors from Q10 that DTD missed to support your answer.

---

## Part 4 — XPath Extraction (Predict, Then Verify)

All XPath queries in this part run against `invoice_valid.xml` (the validated document).

### Q12. Basic path — supplier name

**Predict** the result, then run:

In [ ]:
%%xpath invoice_valid.xml
/Invoice/Supplier/Name/text()


Parse this expression using the axis–node-test–predicate model:
- Axis steps: ?
- Node tests: ?
- Predicates: ?

### Q13. Descendant axis — all line amounts

**Predict** how many results and what values:

In [ ]:
%%xpath invoice_valid.xml
//InvoiceLine/LineAmount/text()


**Question:** Why does `//` (descendant) work here instead of the full path `/Invoice/InvoiceLines/InvoiceLine/LineAmount`? When would you prefer the full path?

### Q14. Predicate filter — items with quantity > 3

**Predict** the result count and values:

In [ ]:
%%xpath invoice_valid.xml
/Invoice/InvoiceLines/InvoiceLine[Quantity > 3]/ItemName/text()


**Verify:** Does the result match the prediction from the narrative checkpoint (Section 3.7)?

### Q15. Attribute extraction — currency codes

**Predict** the result:

In [ ]:
%%xpath invoice_valid.xml
/Invoice/MonetaryTotal/PayableAmount/@currencyID


**Question:** How does `@currencyID` differ from a child element? Could the XSD have modelled currency as a child element `<CurrencyCode>SGD</CurrencyCode>` instead of an attribute? What would change in the XPath?

### Q16. Namespace trap — unqualified path on namespaced document

Run the following XPath on the **namespaced** invoice:

In [ ]:
%%xpath invoice_namespaced.xml
/Invoice/Supplier/Name/text()


**Predict:** Will this return "Kopi Tech Pte Ltd" or empty?

Now run with the namespace bound:

In [ ]:
%%xpath invoice_namespaced.xml
/inv:Invoice/inv:Supplier/inv:Name/text()


(Note: `cellspell` automatically binds prefixes declared in the document.)

**Record:** What changed? In 1–2 sentences, explain why unqualified paths fail on namespaced documents.

---

## Part 5 — JSON Schema Validation (The Modern Equivalent)

This part connects to the narrative's Section 4.7 and resolves Failure 3.

### Q17. Validate with JSON Schema

Run the following cell to validate the malformed payment notification from Failure 3 against the JSON Schema:

In [ ]:
import json
from jsonschema import validate, ValidationError, Draft202012Validator

schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "required": ["amount", "currency", "paymentDate", "merchantId"],
    "properties": {
        "amount":      {"type": "number", "exclusiveMinimum": 0},
        "currency":    {"type": "string", "enum": ["SGD", "USD", "EUR", "GBP"]},
        "paymentDate": {"type": "string", "format": "date"},
        "merchantId":  {"type": "string", "pattern": "^M-\\d{4}$"}
    },
    "additionalProperties": False
}

# The malformed payload from Failure 3
payload = {
    "amount": "150.00",
    "currency": "sgd",
    "payment_date": "2025-02-13",
    "merchant_id": "M-2001"
}

validator = Draft202012Validator(schema)
errors = list(validator.iter_errors(payload))
for e in errors:
    print(f"  Path: {list(e.path) or '(root)'}  |  Error: {e.message}")
print(f"\nTotal errors: {len(errors)}")


**Before running, predict:** How many errors will the validator report? Map each expected error to a specific JSON Schema rule (type, required, enum, additionalProperties).

**After running:** Compare actual errors to your predictions. Which errors match the XSD four-check model from Part 2?

### Q18. Fix and re-validate

Create a corrected payload that passes validation:

In [ ]:
corrected = {
    "amount": 150.00,
    "currency": "SGD",
    "paymentDate": "2025-02-13",
    "merchantId": "M-2001"
}

errors = list(validator.iter_errors(corrected))
if not errors:
    print("✓ Valid")
else:
    for e in errors:
        print(f"  Error: {e.message}")


**Record:** What did you change? Map each fix to the JSON Schema rule it satisfies.

---

## Part 6 — Closing Bridge (Cross-Model Comparison)

### Q19. Complete the comparison table

Using your experience from this lab and Chapters 5–6, fill in the table:

| Dimension | Relational (Ch 5) | JSON / MongoDB (Ch 6) | XML / XSD (Ch 7) | JSON Schema (Ch 7) |
|---|---|---|---|---|
| Schema posture | | | | |
| Enforcement spectrum position | | | | |
| When validation runs | | | | |
| What it catches | | | | |
| Primary context | | | | |
| Who controls the schema? | | | | |

### Q20. Applied scenario — two workloads

A Singapore fintech company has two data flows:

1. **Internal API:** Mobile app sends payment events to the backend (JSON, same team controls both ends, schema evolves weekly).
2. **Regulatory reporting:** Backend submits monthly transaction summaries to MAS (Monetary Authority of Singapore) via an XML portal with a published XSD.

For each flow, recommend:
- (a) Which validation approach? (engine rejects / validator rejects with XSD / validator rejects with JSON Schema / no validation)
- (b) Justify your choice using the trust boundary distinction from Section 4.5.
- (c) What is the cost of choosing the wrong approach? (Reference a specific failure from the narrative.)

---

## Submission Checklist

Before submitting, verify you have:

- [ ] Five completed contract cards (Q1–Q5) with all fields filled
- [ ] Prediction table for invalid invoice errors (Q7) completed **before** running validator
- [ ] XSD validation output recorded with comparison to predictions (Q8)
- [ ] DTD limitation table (Q10) with explanations for what DTD misses
- [ ] DTD limitation analysis paragraph (Q11)
- [ ] Five XPath results with axis–node-test–predicate parsing (Q12–Q16)
- [ ] Namespace trap demonstrated with explanation (Q16)
- [ ] JSON Schema validation with error predictions and comparison (Q17)
- [ ] Corrected JSON payload passing validation (Q18)
- [ ] Cross-model comparison table (Q19)
- [ ] Applied scenario analysis with justified recommendations (Q20)